<a href="https://colab.research.google.com/github/Patrick190508/whaleshark-mafia-island/blob/main/notebooks/OBIS_Dataset_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Raw OBIS to OBIS + Copernicus Dataset


## Step 1 - upload OBIS dataset in the nootebook

In [2]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/Patrick190508/whaleshark-mafia-island/main/data/Occurrence.tsv"
db = pd.read_csv(url, sep="\t", low_memory=False)
print(db.shape)
db.head()

(17224, 282)


,dataset_id,id,acceptedNameUsage,acceptedNameUsageID,accessRights,aphiaid,areas,associatedMedia,associatedOccurrences,associatedOrganisms,...,vernacularName,verticalDatum,vitality,waterBody,wrims,year,originalScientificName,flags,dropped,absence
0,003063a2-57ad-4c7b-a58b-df6f40fbf4ba,07d292ea-1156-4b8a-ad10-7136ed51e783,NaN,NaN,NaN,105847,"[266, 272, 40005, 34288]",NaN,NaN,NaN,...,Whale shark,NaN,NaN,g,True,1993,Rhincodon typus,[NO_DEPTH],False,False
1,031dba90-d480-419a-a562-be9a34bc5c49,7ab66a61-fe45-4370-9343-769a7236ef6c,NaN,NaN,NaN,105847,"[8, 9, 40041, 34364]",NaN,NaN,NaN,...,whale shark,NaN,NaN,NaN,True,NaN,"Rhincodon typus Smith, 1828",[],False,False
2,031dba90-d480-419a-a562-be9a34bc5c49,21b517e3-d309-4c6a-9769-450f786f06cf,NaN,NaN,NaN,105847,"[8, 9, 34364]",NaN,NaN,NaN,...,whale shark,NaN,NaN,NaN,True,NaN,"Rhincodon typus Smith, 1828",[],False,False
3,031dba90-d480-419a-a562-be9a34bc5c49,a9d987cd-157d-45b7-acc4-2de9932dae5b,NaN,NaN,NaN,105847,"[8, 9, 20154, 40040, 34364]",NaN,NaN,NaN,...,whale shark,NaN,NaN,NaN,True,NaN,"Rhincodon typus Smith, 1828",[],False,False
4,031dba90-d480-419a-a562-be9a34bc5c49,609f34f0-0420-4a48-90f9-64afae2486f4,NaN,NaN,NaN,105847,"[8, 9, 20154, 40040, 34364]",NaN,NaN,NaN,...,whale shark,NaN,NaN,NaN,True,NaN,"Rhincodon typus Smith, 1828",[],False,False


## Step 2 - Cleaning the records that are not usable for the research

### 1) Filtering and correcting by Data values

Completing the 4 columns: Date, Year, Month, Day using date_mid (mid values from date_start and date_end)

In [3]:
db['date'] = pd.to_datetime(db['date_mid'], unit='ms')
db['year'] = db['date'].dt.year
db['month'] = db['date'].dt.month
db['day'] = db['date'].dt.day
db[['eventDate', 'date', 'year', 'month', 'day']].head()

,eventDate,date,year,month,day
0,1993-05-30T18:55:42,1993-05-30,1993.0,5.0,30.0
1,2008-01-27,2008-01-27,2008.0,1.0,27.0
2,2012-01-07,2012-01-07,2012.0,1.0,7.0
3,2011-03-19,2011-03-19,2011.0,3.0,19.0
4,2014-11-09,2014-11-09,2014.0,11.0,9.0


Eliminating records with:

*   No Date
*   Too old (befor september 1981)
*   With at least month precision





In [4]:
print("start:", len(db))

db = db[db['date'].notna()]
print("with data:", len(db))

db = db[db['date'] >= '1981-09-01']
print("from september 1981:", len(db))

# Each record has the values date_start & date_end (in millisecond) this range is used not 0 if there is a range of uncertainity for the date of the encounter
day_interval = (db['date_end'] - db['date_start']) / (1000*60*60*24)
# With the variable day_interval we exclude any records which day is not sure (from initial analysis the 75th percentile of day_interval was 0 so few records excluded)
db = db[day_interval <= 1]
print("precise date within day", len(db))

start: 17224
with data: 16976
from september 1981: 16940
precise date within day 16477


**Before Date filtering: 17224 Records**

**After Date filtering: 16477 Records**



---




### 2) Filtering by types of observation (human recordings/telemetry)

As asked by Samuel we will use only Human recordings

In [5]:
print("All records: ", len(db))
db['basisOfRecord'] = db['basisOfRecord'].str.lower()
db = db[db['basisOfRecord'] == 'humanobservation']
print("Only human sightings:", len(db))

All records:  16477
Only human sightings: 10505


**Before Date types filtering: 16477 Records**

**After Date types filtering: 10505 Records**



---



### 3) Eliminating bathymetry errors (records on land)

Since eliminating all the records with negative bathymetry would not consider the error due to the precision of the bathymetry measurement we will insert a thresold using the shoredistance of the sightings and flag the negative-bathymetry records.

In [6]:
print("All records: ", len(db))
on_land = db[db['bathymetry'] <= 0]
on_land = on_land[((on_land['shoredistance']) <= 500)&((on_land['shoredistance']) >= 0)]
on_land = on_land[(on_land['coordinateUncertaintyInMeters'].isna())|(on_land['coordinateUncertaintyInMeters'] <= 300)]
on_sea = db[db['bathymetry'] > 0]
db = pd.concat([on_land, on_sea])
db = db.copy()
db['is_on_land'] = db['bathymetry'] <= 0
print("Only on sea or near coast records: ", len(db))
print("On land sightings: ", len(db[db['is_on_land'] == True]))

All records:  10505
Only on sea or near coast records:  9879
On land sightings:  150


***Run this code if you want to exclude the near coast records***

In [7]:
print("All records: ", len(db))
#db = db[db['bathymetry'] >= 0]
print("Only on sea records: ", len(db))

All records:  9879
Only on sea records:  9879


**Before Date types filtering: 10505 Records**

**After Date types filtering: 9879 Records (150 flagged with is_on_land = True)**

### 4) Analyzing uncertainity in records and marking uncertainity with flags

To train our model we need to establish if a record is precise or not

In [8]:
pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.max_columns', None)
db['coordinateUncertaintyInMeters'].describe()

,coordinateUncertaintyInMeters
count,"1,597.0"
mean,"44,263.4"
std,"168,980.7"
min,0.1
25%,500.0
50%,"30,357.0"
75%,"31,254.0"
max,"2,931,896.0"


**1597/9879 records are obscured**




I will put a thresold eliminate the records with an uncertainity over 40km

In [9]:
print("All records:", len(db))
inc = db['coordinateUncertaintyInMeters']
db['uncertainity'] = inc >= 20000
db = db[~(inc > 40000)]
print("After removing uncertainty > 40 km:", len(db))
print(db['uncertainity'].value_counts())

All records: 9879
After removing uncertainty > 40 km: 9754
uncertainity
False    8823
True      931
Name: count, dtype: int64


**Before obscuring filtering: 9879 Records**


**After obscuring filtering: 9754 Records**


### 5) Removing Duplicated records

Let's check how many duplicated records we have by analyzing the dataset_id, decimalLatitude, decimalLongitude, Date, and catalogNumber (id assigned by the single dataset)

In [10]:
keys = ['dataset_id', 'decimalLatitude', 'decimalLongitude', 'date', 'catalogNumber']
print("Duplicated Records", db.duplicated(subset=keys).sum())

Duplicated Records 9


**9 duplicated records**

In [11]:
db = db.drop_duplicates(subset=keys)
print("Cleaned Dataset", len(db))

Cleaned Dataset 9745


**Before eliminating duplicates: 9754 Records**


**After eliminating duplicates: 9745 Records**


### 6) Removing dead encounter (using the vitality parameters)

In [12]:
print("dead sharks:", (db['vitality'] == 'dead').sum())
db = db[db['vitality'] != 'dead']
print("Records after remooving dead sharks", len(db))

dead sharks: 14
Records after remooving dead sharks 9731


## Step 3 eliminating the empty or not needed columns

### 1) Eliminating Empty columns

In [13]:
print("All the columns:", db.shape[1])
db = db.dropna(axis=1, how="all")
print("Only columns with at least 1 value", db.shape[1])

All the columns: 285
Only columns with at least 1 value 128


**Before Eliminating empty columns: 285 Columns**

**After Eliminating empty columns: 128 Columns**

### 2) Eliminating Constant values columns

Columns that only show one value don't give any information about the records

In [14]:
print("Before Eliminating constant columns:", db.shape[1])
costant_columns = [c for c in db.columns if db[c].nunique(dropna=False) == 1]
db = db.drop(costant_columns, axis=1)
print("After Eliminating constant columns:", db.shape[1])

Before Eliminating constant columns: 128
After Eliminating constant columns: 92


**Before Eliminating constant columns: 128 Columns**

**After Eliminating constant columns: 92 Columns**


### 3) Keeping only useful columns for the Data analysis

In [15]:
keep = ["dataset_id", "id", "occurrenceID", "catalogNumber", "institutionCode", "datasetName", "decimalLatitude", "decimalLongitude", "coordinateUncertaintyInMeters", "date", "date_start", "date_end", "year", "month", "day", "eventTime", "bathymetry", "shoredistance", "sst", "sss", "flags", "is_on_land", "uncertainity"]
db = db[keep]
print(db.shape)


(9731, 23)


Final Dataset Shape after first cleaning process: **9731 rows**, **23 columns**

## Step 4 - Copernicus data insertion

### 1) Installing dependencies and logging in the copernicus account

*Username: psilingardi*

*Password: Patrick1908@*

In [20]:
!pip install -q copernicusmarine
import copernicusmarine
copernicusmarine.login()
sst_ds = copernicusmarine.open_dataset(
    dataset_id="METOFFICE-GLO-SST-L4-REP-OBS-SST",
    variables=["analysed_sst"]
)
print(sst_ds.sizes)

INFO - 2026-09-13T20:07:14Z - Using existing credentials from /root/.copernicusmarine/.copernicusmarine-credentials. Use --force-overwrite combined with credentials (specified by arguments, netrc file or environment variables) to always overwrite.
INFO:copernicusmarine:Using existing credentials from /root/.copernicusmarine/.copernicusmarine-credentials. Use --force-overwrite combined with credentials (specified by arguments, netrc file or environment variables) to always overwrite.
INFO - 2026-09-13T20:07:16Z - Selected dataset version: "202003"
INFO:copernicusmarine:Selected dataset version: "202003"
INFO - 2026-09-13T20:07:16Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


Frozen({'time': 16253, 'latitude': 3600, 'longitude': 7200})


### 2) Inserting SST from Copernicus dataset

In [ ]:
import numpy as np, xarray as xr, os

# Check if there are any progress (help in case of crash)
if os.path.exists("sst_progress.csv"):
    progress = pd.read_csv("sst_progress.csv")
    db['sst_cmems'] = db['id'].map(progress.set_index('id')['sst_cmems'])
    print("restarted:", db['sst_cmems'].notna().sum(), "already done")
else:
    db['sst_cmems'] = np.nan

# --- solo i record ancora da fare, e solo nel periodo coperto dal dataset REP (fino al 2022)
records_to_do = db[db['sst_cmems'].isna() & (db['date'] <= '2022-05-31')]
print("records to do: ", len(records_to_do))

step = 300
for start in range(0, len(records_to_do), step):
    b = records_to_do.iloc[start:start+step]
    pts = sst_ds['analysed_sst'].sel(
        time=xr.DataArray(pd.to_datetime(b['date']).values, dims="p"),
        latitude=xr.DataArray(b['decimalLatitude'].values, dims="p"),
        longitude=xr.DataArray(b['decimalLongitude'].values, dims="p"),
        method="nearest"
    ).values
    db.loc[b.index, 'sst_cmems'] = pts - 273.15
    db[['id', 'sst_cmems']].to_csv("sst_progress.csv", index=False)
    print(f"{start+len(b)}/{len(records_to_do)}", end="  ")

records to do:  9176
